# Swiss Blue-Chip Stock Return Analysis
## Confidence Intervals, Statistical Testing & Market Scenarios

**Scientific Programming – Project Work FS2026**

---

### Research Question
Are the mean daily returns of major Swiss stocks (UBS, Nestlé, Novartis) significantly different from zero, and how do confidence interval widths vary across tickers, sample sizes, and market regimes?

### Objectives
1. Collect real-world stock data for multiple tickers via Yahoo Finance API
2. Compute confidence intervals for mean daily returns
3. Perform hypothesis testing (t-test with p-value) and correlation analysis
4. Compare real data against synthetic market scenarios
5. Store results in an SQLite database
6. Generate LLM-powered summaries of statistical results

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
import sqlite3

from analysis import (
    ticker, TICKERS, log, make_reliability_table, summarize_returns_for_ci,
    plot_reliability_table, build_ci_table, plot_mean_se_and_ci,
    plot_ci_widths, rolling_ci_analysis, plot_rolling_ci_all,
    t_confidence_interval, MeanCI
)
from data_loader import DataLoader, prepare_data
from scenarios import generate_scenario, plot_scenarios_overview_one_image
from database import (
    get_connection, init_db, store_prices, store_ci_results,
    query_prices, query_ci_results, query_avg_return_by_ticker
)

sns.set_context("talk")
plt.rcParams.update({"figure.dpi": 120})

print(f"Analysing tickers: {TICKERS}")

## 2. Data Collection (Web API)
Using `yfinance` to fetch real-world stock data from Yahoo Finance for multiple Swiss blue-chip stocks.

In [ ]:
loader = DataLoader(lookback_days=180)
raw_data = loader.fetch_all()

for sym, df in raw_data.items():
    print(f"{sym}: {df.shape[0]} rows, columns: {list(df.columns)[:5]}")

## 3. Data Preparation
- Column name cleaning with **regular expressions**
- Forward-filling missing values
- Computing daily returns

**Data structures used**: lists, dictionaries, sets, tuples, pandas DataFrames

In [ ]:
prepared = {sym: prepare_data(df, n_trading_days=None) for sym, df in raw_data.items()}

for sym, df in prepared.items():
    print(f"\n{sym}: {len(df)} trading days")
    display(df.describe())

# Python built-in data structures
ticker_set = set(prepared.keys())                    # set
ticker_list = list(prepared.keys())                  # list
first_prices = tuple(df["Price"].iloc[0] for df in prepared.values())  # tuple
summary_dict = {                                     # dictionary
    sym: {
        "mean_return": df["Return"].mean(),
        "std_return": df["Return"].std(),
        "min_price": df["Price"].min(),
        "max_price": df["Price"].max(),
    }
    for sym, df in prepared.items()
}

print(f"\nTicker set: {ticker_set}")
print(f"First prices (tuple): {first_prices}")
for sym, s in summary_dict.items():
    print(f"  {sym}: mean={s['mean_return']:.6f}, std={s['std_return']:.6f}")

## 4. Data Exploration
Tables and visualisations for exploratory analysis.

In [ ]:
fig, axes = plt.subplots(len(TICKERS), 3, figsize=(18, 5 * len(TICKERS)))

for i, sym in enumerate(TICKERS):
    df = prepared[sym]
    axes[i, 0].plot(df.index, df["Price"])
    axes[i, 0].set_title(f"{sym} Price (last 6 months)")
    axes[i, 0].set_ylabel("Price")
    axes[i, 0].grid(True, linestyle="--", alpha=0.5)

    axes[i, 1].hist(df["Return"], bins=30, edgecolor="black", alpha=0.7)
    axes[i, 1].axvline(0, color="red", linestyle="--")
    axes[i, 1].set_title(f"{sym} Return Distribution")
    axes[i, 1].set_xlabel("Daily Return")

    stats.probplot(df["Return"].dropna(), dist="norm", plot=axes[i, 2])
    axes[i, 2].set_title(f"{sym} Q-Q Plot")

plt.tight_layout()
plt.show()

## 5. Reliability Table (Critical t-values)

In [ ]:
confs_table = [0.80, 0.90, 0.95, 0.98, 0.99, 0.998, 0.999]
reliability_tbl = make_reliability_table(confs_table)
display(reliability_tbl)
plot_reliability_table(reliability_tbl, title="Critical t-values (df=9)")

## 6. Confidence Intervals (10 Trading Days) — All Tickers

In [ ]:
for sym in TICKERS:
    stock10 = prepare_data(raw_data[sym], n_trading_days=10)
    stats10 = summarize_returns_for_ci(stock10["Return"])
    print(f"\n{sym}: Mean={stats10.mean:.6f}, SE={stats10.se:.6f}, df={stats10.df}")

    ci_tbl = build_ci_table(stats10, [0.90, 0.95, 0.99])
    display(ci_tbl)
    plot_mean_se_and_ci(stats10, ci_tbl, title_prefix=f"{sym} (10 days)")
    plot_ci_widths(ci_tbl, title=f"{sym} (10 days) – CI Widths")

## 7. Statistical Hypothesis Tests (p-values)

**One-sample t-test**: Is the mean daily return significantly different from 0?
- $H_0$: $\mu = 0$
- $H_1$: $\mu \neq 0$
- Significance level: $\alpha = 0.05$

**Two-sample t-test**: Do UBS and Nestlé have the same mean return?

**Correlation analysis**: Autocorrelation of returns (predictability test)

In [ ]:
print("=" * 60)
print("ONE-SAMPLE T-TESTS (full dataset)")
print("=" * 60)

for sym in TICKERS:
    returns = prepared[sym]["Return"]
    t_stat, p_val = stats.ttest_1samp(returns, popmean=0)
    sig = "*** REJECT H0" if p_val < 0.05 else "fail to reject H0"
    print(f"{sym:10s}: t={t_stat:+.4f}, p={p_val:.6f}  =>  {sig}")

print()
print("=" * 60)
print("TWO-SAMPLE T-TEST: UBS vs Nestle")
print("=" * 60)

t2, p2 = stats.ttest_ind(prepared["UBS"]["Return"], prepared["NESN.SW"]["Return"])
print(f"t-statistic: {t2:.4f}")
print(f"p-value:     {p2:.6f}")
print(f"Result: {'Different means' if p2 < 0.05 else 'No significant difference'} at alpha=0.05")

print()
print("=" * 60)
print("RETURN AUTOCORRELATION (lag-1)")
print("=" * 60)

for sym in TICKERS:
    rets = prepared[sym]["Return"].dropna()
    corr, cp = stats.pearsonr(rets.iloc[1:].values, rets.shift(1).dropna().values)
    sig = "significant" if cp < 0.05 else "not significant"
    print(f"{sym:10s}: r={corr:+.4f}, p={cp:.6f}  =>  {sig}")

print()
print("=" * 60)
print("CROSS-TICKER CORRELATION MATRIX")
print("=" * 60)

ret_matrix = pd.DataFrame({sym: prepared[sym]["Return"] for sym in TICKERS}).dropna()
corr_matrix = ret_matrix.corr()
display(corr_matrix)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap="RdYlGn", center=0, ax=ax, fmt=".3f")
ax.set_title("Return Correlation Matrix")
plt.tight_layout()
plt.show()

## 8. Rolling CI Analysis

In [ ]:
for sym in TICKERS:
    print(f"\nRolling CI for {sym}:")
    confs = [0.90, 0.95, 0.99]
    roll_dfs = {c: rolling_ci_analysis(prepared[sym], window=10, conf=c) for c in confs}
    plot_rolling_ci_all(roll_dfs, ystep=0.05, ylims=(-0.15, 0.20))

## 9. Synthetic Market Scenarios

In [ ]:
primary = TICKERS[0]
stocks_60 = prepare_data(raw_data[primary], n_trading_days=60)
base_price = float(stocks_60["Price"].iloc[0])

scenarios = {
    f"Real {primary}": stocks_60,
    "Bull": generate_scenario("bull", n_days=60, seed=42, base_price=base_price),
    "Bear": generate_scenario("bear", n_days=60, seed=42, base_price=base_price),
    "Volatile": generate_scenario("volatile", n_days=60, seed=42, base_price=base_price),
    "Crisis": generate_scenario("crisis", n_days=60, seed=42, base_price=base_price),
    "Neutral": generate_scenario("neutral", n_days=60, seed=42, base_price=base_price),
}

plot_scenarios_overview_one_image(
    scenarios, conf=[0.90, 0.95, 0.99], sample_n=10,
    title=f"{primary} Real Data vs. Synthetic Market Scenarios",
    rebase_to=base_price
)

## 10. SQLite Database Storage
Storing data and results in an SQLite database and querying them back with SQL.

In [ ]:
conn = get_connection()
init_db(conn)

for sym in TICKERS:
    n = store_prices(prepared[sym], sym, conn)
    print(f"Stored {n} price rows for {sym}")

primary_10 = prepare_data(raw_data[TICKERS[0]], n_trading_days=10)
primary_stats = summarize_returns_for_ci(primary_10["Return"])
primary_ci = build_ci_table(primary_stats, [0.90, 0.95, 0.99])
store_ci_results(primary_ci, TICKERS[0],
                 window_start=str(primary_10.index.min().date()),
                 window_end=str(primary_10.index.max().date()),
                 conn=conn)
print("Stored CI results")

print("\n--- Average return by ticker (SQL aggregate query) ---")
display(query_avg_return_by_ticker(conn=conn))

print("\n--- Days with |return| > 3% (SQL filter query) ---")
extreme = pd.read_sql_query(
    "SELECT ticker, date, price, return_ FROM stock_prices "
    "WHERE ABS(return_) > 0.03 ORDER BY ABS(return_) DESC",
    conn
)
display(extreme if len(extreme) > 0 else "No extreme days found")

conn.close()

## 11. LLM-Powered Analysis Summary
Using OpenAI API to generate natural-language interpretations of statistical results.

> **Note:** Set `OPENAI_API_KEY` environment variable before running this cell.
> If not available, this cell will show a fallback message.

In [ ]:
try:
    from llm_summary import summarise_statistics, compare_tickers_summary

    # Single-ticker summary
    primary_returns = prepared[TICKERS[0]]["Return"]
    t_stat, p_val = stats.ttest_1samp(primary_returns, popmean=0)
    primary_10 = prepare_data(raw_data[TICKERS[0]], n_trading_days=10)
    primary_stats = summarize_returns_for_ci(primary_10["Return"])
    primary_ci = build_ci_table(primary_stats, [0.90, 0.95, 0.99])

    summary = summarise_statistics(
        ticker=TICKERS[0],
        descriptive_stats=prepared[TICKERS[0]].describe(),
        t_test_result=(t_stat, p_val),
        ci_table=primary_ci,
    )
    print("=== LLM Summary (single ticker) ===")
    print(summary)

    # Multi-ticker comparison
    ticker_stats = {}
    for sym in TICKERS:
        r = prepared[sym]["Return"]
        ts, pv = stats.ttest_1samp(r, popmean=0)
        ticker_stats[sym] = {"mean": r.mean(), "std": r.std(), "t_stat": ts, "p_value": pv}

    comparison = compare_tickers_summary(ticker_stats)
    print("\n=== LLM Comparison (all tickers) ===")
    print(comparison)

except RuntimeError as e:
    print(f"LLM unavailable: {e}")
    print("Set OPENAI_API_KEY to enable LLM summaries.")

## 12. Conclusions

### Key Findings
1. **Confidence intervals** widen with increasing confidence level
2. **Rolling analysis** reveals time-varying uncertainty in return estimates
3. **Hypothesis tests** show whether Swiss blue-chip returns differ from zero
4. **Cross-ticker correlation** reveals co-movement between UBS, Nestlé, and Novartis
5. **Scenario comparison**: Volatile/crisis scenarios produce wider CIs

### Methods Used
- Real-world data collection (yfinance Web API) — multiple tickers
- Data preparation with regex, pandas
- Python data structures: lists, dicts, sets, tuples, DataFrames
- Conditional statements, loops, procedural & OOP programming
- Statistical visualisations (matplotlib, seaborn)
- Statistical analysis with p-values (scipy): t-tests, correlation
- SQLite database with SQL queries
- LLM integration (OpenAI API) for automated analysis summaries
- Streamlit web application (`streamlit run app.py`)